# Lekcija 08 - Višeagenta dizajnerski obrazac


## Postavljanje


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Zašto sustavi s više agenata?

Zadaci iz stvarnog svijeta poput planiranja putovanja uključuju mnogo različitih vrsta stručnosti — logistiku, lokalno znanje, budžetiranje i drugo. Jedan agent koji pokušava sve obraditi brzo postaje nezgrapan.

Sustavi s više agenata rješavaju to kroz **specijalizaciju**: svaki agent se fokusira na jedno područje stručnosti, proizodeći kvalitetnije rezultate nego generalist. Također poboljšavaju **skalabilnost** — možete dodavati nove agente (npr. stručnjaka za letove, kritičara restorana) bez prepisivanja postojećeg toka rada. Agenti se povezuju kroz strukturirani tijek, prenoseći kontekst od jednog do drugog.


## Izrada specijaliziranih agenata


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Izgradnja sekvencijalnog toka rada

`WorkflowBuilder` vam omogućuje povezivanje agenata u usmjereni graf. Ovdje stvaramo jednostavnu dvofaznu cjevovod: **TravelPlanner** sastavlja plan puta, zatim ga **TravelConcierge** pregledava i poboljšava.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Dodavanje više agenata u tijek rada

Jedna od najvećih prednosti obrazca više agenata je koliko je lako proširiti ga. Ispod dodajemo agenta **BudgetReviewer** koji provjerava plan prema proračunu putnika, označava stavke koje bi mogle povećati troškove preko limita i predlaže alternative za uštedu novca. Tijek rada sada pokreće tri agenta zaredom:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Sažetak

U ovoj lekciji naučili ste kako:

1. **Kreirati specijalizirane agente** — svaki s fokusiranom ulogom (planiranje, konsijerž, pregled budžeta).
2. **Povezati agente u sekvencijalni tijek rada** koristeći `WorkflowBuilder` i `add_edge`.
3. **Prijenos izlaza u stvarnom vremenu** iz lančanog višestrukog agenta, prateći koji agent govori.
4. **Proširiti tijek rada** dodavanjem novih agenata u lanac bez mijenjanja postojećih.

Dizajn obrazac višestrukih agenata održava svaki agenta jednostavnim, istovremeno proizvodeći bogatije, temeljitije pregledane rezultate nego što bi to mogao postići bilo koji pojedinačni agent samostalno.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:  
Ovaj dokument je preveden koristeći AI uslugu za prevođenje [Co-op Translator](https://github.com/Azure/co-op-translator). Iako nastojimo postići točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba se smatrati autoritativnim izvorom. Za važne informacije preporučuje se profesionalan ljudski prijevod. Ne odgovaramo za bilo kakva nesporazuma ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
